# LeetCode 3206 — Alternating Groups I bằng Python

Link đề: https://leetcode.com/problems/alternating-groups-i/ · Độ khó: **Easy** · Chủ đề: Array, Sliding Window (mảng vòng tròn)

Skeleton LeetCode:

```python
class Solution:
    def numberOfAlternatingGroups(self, colors: List[int]) -> int:
        ...
```

Ghi chú: đề và các ví dụ được viết theo hiểu biết về đề gốc (đề gốc có kèm hình minh họa vòng tròn gạch); mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: một **vòng tròn** các viên gạch đỏ/xanh (`0`/`1`); cứ **3 viên liên tiếp** mà viên **giữa khác màu cả hai viên bên cạnh** thì gọi là một **nhóm xen kẽ**; đếm số nhóm đó. Viên đầu và viên cuối nằm **cạnh nhau**.
2. Nắm các công cụ Python: chỉ số vòng bằng phép `%`, chỉ số âm `colors[-1]`, cắt lát `colors[:2]` và nối list `colors + colors[:2]`, `zip` ba list lệch nhau, và đếm điều kiện bằng `sum(điều_kiện for ...)`.
3. Đi từ **kiểm tra từng vị trí bằng `%`** → **kéo dài mảng để bỏ `%`** → **mảng "cạnh khác màu"** → **trải vòng thành `2n` và đếm đoạn xen kẽ đang chạy** → **bản `zip` một dòng**.
4. Hiểu mấu chốt: chỉ có **hai màu**, nên "giữa khác hai bên" nghĩa là **hai bên bằng nhau và khác giữa** (`a == c != b`); và vòng tròn chỉ là mảng thẳng mà **chỉ số được lấy `% n`**.
5. Tránh các lỗi kinh điển: **quên vòng** (không xét nhóm đi qua điểm nối đầu-cuối), **`IndexError`** khi đọc `colors[i + 1]` ở cuối mảng, kiểm tra thiếu điều kiện "khác giữa", và **đếm trùng** khi trải vòng thành `2n`.

## 2. Đề bài & ví dụ

Có một **vòng tròn** các viên gạch đỏ và xanh. Cho mảng số nguyên `colors`: viên gạch thứ `i` có màu `colors[i]`.

- `colors[i] == 0` nghĩa là viên `i` màu **đỏ**.
- `colors[i] == 1` nghĩa là viên `i` màu **xanh**.

Cứ **3 viên liên tiếp** trong vòng tròn mà **màu xen kẽ** (viên **giữa** khác màu viên **bên trái** và khác màu viên **bên phải**) thì gọi là một **nhóm xen kẽ**.

Hãy trả về **số nhóm xen kẽ**.

**Lưu ý:** vì `colors` biểu diễn một **vòng tròn**, viên **đầu tiên** và viên **cuối cùng** được coi là **nằm cạnh nhau**.

**Ví dụ 1**

- Input: `colors = [1,1,1]`
- Output: `0`
- Giải thích: ba viên cùng màu, không có viên giữa nào khác hai bên.

**Ví dụ 2**

- Input: `colors = [0,1,0,0,1]`
- Output: `3`
- Giải thích: ba nhóm là `(colors[0], colors[1], colors[2]) = (0,1,0)`, `(colors[3], colors[4], colors[0]) = (0,1,0)` (đi qua điểm nối đầu-cuối) và `(colors[4], colors[0], colors[1]) = (1,0,1)`.

**Ràng buộc:** `3 <= colors.length <= 100`; `0 <= colors[i] <= 1`.

In [1]:
from typing import List
import random

examples = [
    ([1, 1, 1], 0),
    ([0, 1, 0, 0, 1], 3),
]
for colors, expected in examples:
    print("colors =", colors, "-> kỳ vọng", expected)

colors = [1, 1, 1] -> kỳ vọng 0
colors = [0, 1, 0, 0, 1] -> kỳ vọng 3


## 3. Kiến thức nền

### 3.1. Vòng tròn: chỉ số `% n` và chỉ số âm

Một **vòng tròn** có `n` viên đánh số `0..n-1`, viên cạnh viên `n-1` (theo chiều tiến) là viên `0`. Trong code ta lấy chỉ số bằng **phép chia dư** `(i + 1) % n`: khi `i = n - 1` ta được `0`, còn các `i` khác thì không đổi. Muốn lấy viên **trước** viên `0`, Python cho phép **chỉ số âm**: `colors[-1]` là viên cuối, `colors[i - 1]` với `i = 0` chính là viên cuối.

In [2]:
colors = [0, 1, 0, 0, 1]
n = len(colors)
for i in range(n):
    print(f"i={i}: trái = colors[{i - 1}] = {colors[i - 1]}, giữa = {colors[i]}, phải = colors[{(i + 1) % n}] = {colors[(i + 1) % n]}")

i=0: trái = colors[-1] = 1, giữa = 0, phải = colors[1] = 1
i=1: trái = colors[0] = 0, giữa = 1, phải = colors[2] = 0
i=2: trái = colors[1] = 1, giữa = 0, phải = colors[3] = 0
i=3: trái = colors[2] = 0, giữa = 0, phải = colors[4] = 1
i=4: trái = colors[3] = 0, giữa = 1, phải = colors[0] = 0


### 3.2. Cắt lát `colors[:2]` và nối list `colors + colors[:2]`

`colors[:2]` là **2 phần tử đầu**. `colors + colors[:2]` là list mới với **hai phần tử đầu được dán vào cuối**. Nhờ đó khi đọc ba viên liên tiếp `colors2[i], colors2[i+1], colors2[i+2]` ta không cần `%` nữa: phần "quấn vòng" đã được sao chép sẵn.

In [3]:
colors = [0, 1, 0, 0, 1]
extended = colors + colors[:2]
print(colors[:2])                 # [0, 1]
print(extended)                   # [0, 1, 0, 0, 1, 0, 1]
print(len(extended) - 2)          # 5: có đúng n cửa sổ 3 viên liên tiếp

[0, 1]
[0, 1, 0, 0, 1, 0, 1]
5


### 3.3. Với hai màu: "giữa khác cả hai bên" ⇔ "hai bên bằng nhau và khác giữa"

Gọi ba viên liên tiếp là `a, b, c`. Nhóm xen kẽ cần `b != a` và `b != c`. Vì chỉ có hai màu (`0`/`1`), "khác `b`" chính là "bằng màu còn lại" nên `a` và `c` **cùng là màu còn lại**, tức `a == c`. Vậy điều kiện tương đương là `a == c and a != b`. Hãy liệt kê cả `8` bộ ba để thấy.

In [4]:
from itertools import product

for a, b, c in product((0, 1), repeat=3):
    alt1 = (b != a) and (b != c)
    alt2 = (a == c) and (a != b)
    print((a, b, c), "xen kẽ?" , alt1, alt2)
    assert alt1 == alt2

(0, 0, 0) xen kẽ? False False
(0, 0, 1) xen kẽ? False False
(0, 1, 0) xen kẽ? True True
(0, 1, 1) xen kẽ? False False
(1, 0, 0) xen kẽ? False False
(1, 0, 1) xen kẽ? True True
(1, 1, 0) xen kẽ? False False
(1, 1, 1) xen kẽ? False False


### 3.4. `zip` ba list lệch nhau và đếm bằng `sum`

`zip(x, y, z)` ghép các phần tử cùng vị trí: nếu `y` là `x` **dịch trái 1** và `z` là `x` **dịch trái 2** thì mỗi bộ `(p, q, r)` là ba viên liên tiếp. `sum(điều_kiện for ...)` đếm số lần điều kiện `True` (`True` đếm là `1`).

In [5]:
colors = [0, 1, 0, 0, 1]
left = colors
mid = colors[1:] + colors[:1]       # dịch trái 1 (quấn vòng)
right = colors[2:] + colors[:2]     # dịch trái 2 (quấn vòng)
print(list(zip(left, mid, right)))
print(sum(a != b and b != c for a, b, c in zip(left, mid, right)))      # 3

[(0, 1, 0), (1, 0, 0), (0, 0, 1), (0, 1, 0), (1, 0, 1)]
3


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Với `colors = [0, 1, 0, 0, 1]` (`n = 5`), có **5 vị trí bắt đầu** `i = 0..4`; nhóm bắt đầu ở `i` gồm các viên `i, i+1, i+2` (lấy `% 5`):

| bắt đầu `i` | ba viên (chỉ số) | ba màu | giữa khác hai bên? |
|---|---|---|---|
| 0 | 0, 1, 2 | 0, 1, 0 | **có** |
| 1 | 1, 2, 3 | 1, 0, 0 | không (giữa `0` bằng phải `0`) |
| 2 | 2, 3, 4 | 0, 0, 1 | không (giữa `0` bằng trái `0`) |
| 3 | 3, 4, 0 | 0, 1, 0 | **có** (đi qua điểm nối đầu-cuối) |
| 4 | 4, 0, 1 | 1, 0, 1 | **có** (đi qua điểm nối đầu-cuối) |

Có `3` nhóm ⇒ đáp án `3`. Điểm cần chú ý: hai nhóm cuối **đi qua chỗ nối** giữa viên `4` và viên `0`; nếu chỉ xét các bộ ba "thẳng hàng" trong mảng ta chỉ thấy `3` ứng viên (bắt đầu `0, 1, 2`) và đếm thiếu.

In [6]:
colors = [0, 1, 0, 0, 1]
n = len(colors)
count = 0
for i in range(n):
    a, b, c = colors[i], colors[(i + 1) % n], colors[(i + 2) % n]
    ok = b != a and b != c
    count += ok
    print(f"bắt đầu i={i}: ({a}, {b}, {c}) -> {'xen kẽ' if ok else 'không'}")
print("tổng =", count)

bắt đầu i=0: (0, 1, 0) -> xen kẽ
bắt đầu i=1: (1, 0, 0) -> không
bắt đầu i=2: (0, 0, 1) -> không
bắt đầu i=3: (0, 1, 0) -> xen kẽ
bắt đầu i=4: (1, 0, 1) -> xen kẽ
tổng = 3


## 5. Cách 1 — Brute Force / baseline: xét từng vị trí, lấy chỉ số bằng `%`

**Ý tưởng.** Với mỗi `i` trong `0..n-1`, lấy ba viên `colors[i]`, `colors[(i+1) % n]`, `colors[(i+2) % n]` rồi kiểm tra viên giữa khác cả hai bên. Có `n` nhóm cần xét (mỗi vị trí bắt đầu là một nhóm), bám sát đề.

In [7]:
def alt_groups_modulo(colors):
    n = len(colors)
    count = 0
    for i in range(n):
        a = colors[i]
        b = colors[(i + 1) % n]
        c = colors[(i + 2) % n]
        if b != a and b != c:
            count += 1
    return count


for colors, expected in examples:
    print(alt_groups_modulo(colors), "(kỳ vọng", expected, ")")

0 (kỳ vọng 0 )
3 (kỳ vọng 3 )


**Độ phức tạp (Cách 1).** Gọi `n = len(colors)`. Mỗi vị trí `O(1)` ⇒ **Time `O(n)`**, **Space `O(1)`**. Với `n <= 100` thì tức thì.

**Nút thắt:** không có nút thắt về tốc độ (đã `O(n)`); điểm khó là **chỉ số `%` rối mắt** và dễ quên. Các cách sau làm rõ phần "vòng tròn" theo những cách khác nhau, và cách cuối tổng quát hóa được cho nhóm dài `k` (bài II).

## 6. Ý tưởng thuật toán / bước đột phá

Có hai cách nhìn gọn hơn:

**(a) Biến vòng thành đường thẳng bằng cách dán thêm đầu mảng.** Thay vì `%`, tạo `ext = colors + colors[:2]`; khi đó `ext[i], ext[i+1], ext[i+2]` với `i = 0..n-1` đúng là mọi nhóm của vòng tròn, và chỉ số không bao giờ vượt biên.

**(b) Đếm đoạn xen kẽ đang chạy.** Duyệt mảng và giữ `cnt` = độ dài đoạn xen kẽ liên tiếp đang kết thúc ở vị trí hiện tại (gặp hai viên **bằng nhau** thì đoạn bị cắt, `cnt = 1`; ngược lại `cnt += 1`). Một nhóm 3 viên xen kẽ kết thúc ở vị trí này khi `cnt >= 3`. Để vòng tròn chuẩn, ta **trải vòng thành `2n`** và chỉ đếm khi vị trí `i >= n` (mỗi viên đóng vai "viên cuối của nhóm" đúng **một lần**).

Cách (b) mở rộng được cho nhóm dài `k` chỉ bằng đổi `cnt >= 3` thành `cnt >= k`.

In [8]:
colors = [0, 1, 0, 0, 1]
ext = colors + colors[:2]
print(ext)
groups = [(ext[i], ext[i + 1], ext[i + 2]) for i in range(len(colors))]
print(groups)
print(sum(b != a and b != c for a, b, c in groups))          # 3

[0, 1, 0, 0, 1, 0, 1]
[(0, 1, 0), (1, 0, 0), (0, 0, 1), (0, 1, 0), (1, 0, 1)]
3


## 7. Các cách tiếp theo

### Cách 2 — Kéo dài mảng `colors + colors[:2]` rồi trượt cửa sổ 3 viên

**Giải thích từng đoạn.**

1. `ext = colors + colors[:2]`: hai viên đầu được dán vào cuối để "đóng" vòng.
2. `for i in range(n)`: mỗi `i` là điểm bắt đầu của một nhóm; chỉ số `i + 2 <= n + 1` luôn hợp lệ.
3. `ext[i+1] != ext[i] and ext[i+1] != ext[i+2]` là điều kiện nhóm xen kẽ.

In [9]:
def alt_groups_extended(colors):
    n = len(colors)
    ext = colors + colors[:2]
    count = 0
    for i in range(n):
        if ext[i + 1] != ext[i] and ext[i + 1] != ext[i + 2]:
            count += 1
    return count


for colors, expected in examples:
    print(alt_groups_extended(colors), "(kỳ vọng", expected, ")")

0 (kỳ vọng 0 )
3 (kỳ vọng 3 )


In [10]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionExtended:
    def numberOfAlternatingGroups(self, colors: List[int]) -> int:
        n = len(colors)
        ext = colors + colors[:2]
        count = 0
        for i in range(n):
            if ext[i + 1] != ext[i] and ext[i + 1] != ext[i + 2]:
                count += 1
        return count


print(SolutionExtended().numberOfAlternatingGroups([0, 1, 0, 0, 1]))      # 3

3


**Độ phức tạp Cách 2:** **Time `O(n)`**, **Space `O(n)`** (list `ext` dài `n + 2`).

### Cách 3 — Mảng "cạnh khác màu" `d`

**Giải thích từng đoạn.**

1. Mỗi **cạnh** giữa hai viên liên tiếp `i` và `(i + 1) % n` là "khác màu" nếu `colors[i] != colors[(i+1) % n]`. Đặt `d[i] = 1` nếu cạnh `i` khác màu, ngược lại `0`. Có `n` cạnh (vòng tròn).
2. Một nhóm bắt đầu ở `i` (viên `i, i+1, i+2`) có **hai cạnh**: cạnh `i` (giữa `i` và `i+1`) và cạnh `i+1` (giữa `i+1` và `i+2`).
3. Nhóm xen kẽ ⇔ **cả hai cạnh khác màu** ⇔ `d[i] == 1 and d[(i+1) % n] == 1`. Vậy đáp án là số vị trí `i` mà `d[i]` và `d[i+1]` đều bằng `1`.

Cách này tách bài thành hai việc nhỏ: (1) đánh dấu cạnh, (2) đếm cặp cạnh liên tiếp cùng bằng `1`.

In [11]:
def alt_groups_edges(colors):
    n = len(colors)
    d = [int(colors[i] != colors[(i + 1) % n]) for i in range(n)]      # d[i] = 1 nếu cạnh i khác màu
    return sum(d[i] and d[(i + 1) % n] for i in range(n))


for colors, expected in examples:
    print(alt_groups_edges(colors), "(kỳ vọng", expected, ")")
print([int([0, 1, 0, 0, 1][i] != [0, 1, 0, 0, 1][(i + 1) % 5]) for i in range(5)])     # d của ví dụ 2: [1, 1, 0, 1, 1]

0 (kỳ vọng 0 )
3 (kỳ vọng 3 )
[1, 1, 0, 1, 1]


In [12]:
# Code nộp LeetCode cho Cách 3
class SolutionEdges:
    def numberOfAlternatingGroups(self, colors: List[int]) -> int:
        n = len(colors)
        d = [colors[i] != colors[(i + 1) % n] for i in range(n)]     # True nếu cạnh i khác màu
        return sum(d[i] and d[(i + 1) % n] for i in range(n))


print(SolutionEdges().numberOfAlternatingGroups([1, 1, 1]))      # 0

0


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(n)`** (mảng `d`).

### Cách 4 — Trải vòng thành `2n`, đếm đoạn xen kẽ đang chạy (`cnt`)

**Giải thích từng đoạn.**

1. `k = 3` là độ dài nhóm. `cnt` là độ dài đoạn xen kẽ liên tiếp **kết thúc** ở vị trí `i`.
2. Duyệt `i` từ `0` đến `2n - 1`, viên ở vị trí `i` là `colors[i % n]`, viên trước là `colors[(i - 1) % n]`.
3. Nếu `i > 0` và hai viên **bằng nhau** thì `cnt = 1` (bắt đầu đoạn mới), ngược lại `cnt += 1`.
4. Khi `i >= n` và `cnt >= k`: có một nhóm xen kẽ kết thúc ở viên `i % n`. Mỗi viên `0..n-1` được xét đúng một lần trong nửa sau, nên không đếm trùng.

In [13]:
def alt_groups_unroll(colors, k=3):
    n = len(colors)
    ans = cnt = 0
    for i in range(2 * n):
        if i > 0 and colors[i % n] == colors[(i - 1) % n]:
            cnt = 1                      # hai viên bằng nhau: đoạn xen kẽ bị cắt
        else:
            cnt += 1
        if i >= n and cnt >= k:          # nửa sau: mỗi viên là "viên cuối" của một nhóm đúng một lần
            ans += 1
    return ans


for colors, expected in examples:
    print(alt_groups_unroll(colors), "(kỳ vọng", expected, ")")

0 (kỳ vọng 0 )
3 (kỳ vọng 3 )


In [14]:
# Code nộp LeetCode cho Cách 4
class SolutionUnroll:
    def numberOfAlternatingGroups(self, colors: List[int]) -> int:
        k = 3
        n = len(colors)
        ans = cnt = 0
        for i in range(2 * n):
            if i > 0 and colors[i % n] == colors[(i - 1) % n]:
                cnt = 1
            else:
                cnt += 1
            if i >= n and cnt >= k:
                ans += 1
        return ans


print(SolutionUnroll().numberOfAlternatingGroups([0, 1, 0, 0, 1]))      # 3

3


**Độ phức tạp Cách 4:** **Time `O(n)`** (`2n` bước), **Space `O(1)`**. Đây là cách **dễ mở rộng** sang nhóm `k` viên (bài II).

### Cách 5 — `zip` ba list dịch trái, một dòng

**Giải thích từng đoạn.**

1. `colors`, `colors[1:] + colors[:1]` (dịch trái 1), `colors[2:] + colors[:2]` (dịch trái 2) là ba "lớp" xếp chồng: cùng vị trí là ba viên liên tiếp.
2. `zip` ghép ba lớp; `sum` đếm số bộ có giữa khác hai bên.

Cách này không dùng chỉ số, chỉ cần nhớ rằng "dịch trái" + "quấn đầu về cuối" tạo vòng tròn.

In [15]:
def alt_groups_zip(colors):
    mid = colors[1:] + colors[:1]
    right = colors[2:] + colors[:2]
    return sum(a != b and b != c for a, b, c in zip(colors, mid, right))


for colors, expected in examples:
    print(alt_groups_zip(colors), "(kỳ vọng", expected, ")")

0 (kỳ vọng 0 )
3 (kỳ vọng 3 )


In [16]:
# Code nộp LeetCode cho Cách 5
class SolutionZip:
    def numberOfAlternatingGroups(self, colors: List[int]) -> int:
        mid = colors[1:] + colors[:1]
        right = colors[2:] + colors[:2]
        return sum(a != b and b != c for a, b, c in zip(colors, mid, right))


print(SolutionZip().numberOfAlternatingGroups([0, 1, 0, 0, 1]))      # 3

3


**Độ phức tạp Cách 5:** **Time `O(n)`**, **Space `O(n)`** (hai list dịch).

## 8. TRACE từng bước

Ta trace **Cách 4** (trải vòng thành `2n`) trên `colors = [0, 1, 0, 0, 1]` (`n = 5`, `k = 3`). Mỗi dòng ghi: vị trí `i`, viên hiện tại `colors[i % n]`, viên trước, `cnt` sau cập nhật, có `i >= n` không, và `ans`.

| `i` | viên hiện tại | viên trước | cập nhật `cnt` | `cnt` | `i >= n` và `cnt >= 3`? | `ans` |
|---|---|---|---|---|---|---|
| 0 | 0 | (chưa có) | `+= 1` | 1 | không | 0 |
| 1 | 1 | 0 | khác nhau, `+= 1` | 2 | không | 0 |
| 2 | 0 | 1 | khác nhau, `+= 1` | 3 | không (`i < n`) | 0 |
| 3 | 0 | 0 | **bằng nhau**, `cnt = 1` | 1 | không | 0 |
| 4 | 1 | 0 | khác nhau, `+= 1` | 2 | không | 0 |
| 5 | 0 | 1 | khác nhau, `+= 1` | 3 | **có** | 1 |
| 6 | 1 | 0 | khác nhau, `+= 1` | 4 | **có** | 2 |
| 7 | 0 | 1 | khác nhau, `+= 1` | 5 | **có** | 3 |
| 8 | 0 | 0 | **bằng nhau**, `cnt = 1` | 1 | không | 3 |
| 9 | 1 | 0 | khác nhau, `+= 1` | 2 | không | 3 |

Các nhóm đếm được kết thúc ở viên `5 % 5 = 0`, `1`, `2`: tức nhóm `(3, 4, 0)`, `(4, 0, 1)`, `(0, 1, 2)` — đúng ba nhóm ở mục 4. Đáp án `3`.

In [17]:
def trace_unroll(colors, k=3):
    n = len(colors)
    ans = cnt = 0
    print(f"colors={colors}, n={n}, k={k}")
    for i in range(2 * n):
        cur = colors[i % n]
        prev = colors[(i - 1) % n] if i > 0 else "(chưa có)"
        if i > 0 and cur == prev:
            cnt = 1
            action = "bằng nhau -> cnt = 1"
        else:
            cnt += 1
            action = "cnt += 1"
        counted = i >= n and cnt >= k
        if counted:
            ans += 1
        print(f"i={i}, cur={cur}, prev={prev}, {action}, cnt={cnt}, đếm? {counted}, ans={ans}")
    return ans


assert trace_unroll([0, 1, 0, 0, 1]) == 3

colors=[0, 1, 0, 0, 1], n=5, k=3
i=0, cur=0, prev=(chưa có), cnt += 1, cnt=1, đếm? False, ans=0
i=1, cur=1, prev=0, cnt += 1, cnt=2, đếm? False, ans=0
i=2, cur=0, prev=1, cnt += 1, cnt=3, đếm? False, ans=0
i=3, cur=0, prev=0, bằng nhau -> cnt = 1, cnt=1, đếm? False, ans=0
i=4, cur=1, prev=0, cnt += 1, cnt=2, đếm? False, ans=0
i=5, cur=0, prev=1, cnt += 1, cnt=3, đếm? True, ans=1
i=6, cur=1, prev=0, cnt += 1, cnt=4, đếm? True, ans=2
i=7, cur=0, prev=1, cnt += 1, cnt=5, đếm? True, ans=3
i=8, cur=0, prev=0, bằng nhau -> cnt = 1, cnt=1, đếm? False, ans=3
i=9, cur=1, prev=0, cnt += 1, cnt=2, đếm? False, ans=3


In [18]:
def trace_edges(colors):
    n = len(colors)
    d = [int(colors[i] != colors[(i + 1) % n]) for i in range(n)]
    print("colors =", colors)
    print("d (cạnh khác màu) =", d)
    total = 0
    for i in range(n):
        both = d[i] and d[(i + 1) % n]
        total += both
        print(f"i={i}: d[{i}]={d[i]}, d[{(i + 1) % n}]={d[(i + 1) % n]} -> {'nhóm xen kẽ' if both else 'không'}, tổng={total}")
    return total


assert trace_edges([0, 1, 0, 0, 1]) == 3

colors = [0, 1, 0, 0, 1]
d (cạnh khác màu) = [1, 1, 0, 1, 1]
i=0: d[0]=1, d[1]=1 -> nhóm xen kẽ, tổng=1
i=1: d[1]=1, d[2]=0 -> không, tổng=1
i=2: d[2]=0, d[3]=1 -> không, tổng=1
i=3: d[3]=1, d[4]=1 -> nhóm xen kẽ, tổng=2
i=4: d[4]=1, d[0]=1 -> nhóm xen kẽ, tổng=3


Bảng cho **Cách 3** trên cùng ví dụ: `d = [1, 1, 0, 1, 1]` (cạnh `2`, giữa hai viên `0, 0`, là cạnh duy nhất cùng màu). Cặp cạnh liên tiếp `(d[i], d[i+1])`: `(1,1)` ✓, `(1,0)` ✗, `(0,1)` ✗, `(1,1)` ✓, `(1,1)` ✓ (`i = 4`: `d[4]` và `d[0]`, đi qua chỗ nối) ⇒ `3`.

## 9. Code nộp LeetCode

Bản nộp dùng **Cách 2** (kéo dài mảng, không cần `%` trong thân vòng lặp): rất dễ đọc và `O(n)`.

In [19]:
class Solution:
    def numberOfAlternatingGroups(self, colors: List[int]) -> int:
        n = len(colors)
        ext = colors + colors[:2]                 # dán hai viên đầu vào cuối để đóng vòng
        count = 0
        for i in range(n):                        # mỗi i là điểm bắt đầu một nhóm 3 viên
            if ext[i + 1] != ext[i] and ext[i + 1] != ext[i + 2]:
                count += 1                        # viên giữa khác cả hai bên
        return count


sol = Solution()
for colors, expected in examples:
    print(colors, "->", sol.numberOfAlternatingGroups(colors), "(kỳ vọng", expected, ")")

[1, 1, 1] -> 0 (kỳ vọng 0 )
[0, 1, 0, 0, 1] -> 3 (kỳ vọng 3 )


## 10. Vì sao thuật toán đúng

- **Mỗi nhóm có đúng một vị trí bắt đầu.** Một nhóm là ba viên liên tiếp `i, i+1, i+2` (chỉ số theo `% n`). Vì `n >= 3`, ba chỉ số đó khác nhau và `i = 0..n-1` liệt kê mọi nhóm đúng một lần. Ta đếm **theo vị trí**, nên không đếm trùng.
- **Điều kiện đúng.** Viên giữa khác cả viên trái lẫn viên phải: `b != a and b != c`. Điều kiện này không đổi khi chỉ có hai màu: nó tương đương `a == c != b`.
- **Vòng đóng đúng.** `ext = colors + colors[:2]` chứa `colors[n] = colors[0]` và `colors[n+1] = colors[1]`, nên các nhóm đi qua điểm nối (bắt đầu ở `n-2` và `n-1`) cũng được lấy đúng, tương đương `% n`.
- **Cách 3.** Nhóm xen kẽ ⇔ hai cạnh liên tiếp đều khác màu; số nhóm là số cặp cạnh liên tiếp `(d[i], d[i+1 mod n])` cùng bằng `1`.
- **Cách 4.** `cnt` là độ dài đoạn xen kẽ dài nhất kết thúc ở `i`; có nhóm 3 viên xen kẽ kết thúc ở `i` ⇔ `cnt >= 3`. Trong nửa sau mỗi viên của vòng xuất hiện đúng một lần, và đoạn tối thiểu cần xét (3 viên cuối) nằm trọn trong phần đã trải (vì `n >= 3`).

## 11. Phân tích Time / Space Complexity

Gọi `n = len(colors)` (`3 <= n <= 100`).

| Cách | Time | Space |
|---|---|---|
| 1. Chỉ số `%` | `O(n)` | `O(1)` |
| 2. Kéo dài mảng `+2` | `O(n)` | `O(n)` |
| 3. Mảng cạnh khác màu | `O(n)` | `O(n)` |
| 4. Trải `2n`, đếm `cnt` | `O(n)` | `O(1)` |
| 5. `zip` ba list dịch | `O(n)` | `O(n)` |

Mọi cách đều tuyến tính; chọn theo độ dễ đọc, hoặc theo khả năng mở rộng (Cách 4).

## 12. Bộ test

Ta gom mọi cách vào `impls` rồi so với kỳ vọng.

In [20]:
impls = {
    "modulo": alt_groups_modulo,
    "extended": alt_groups_extended,
    "edges": alt_groups_edges,
    "unroll": alt_groups_unroll,
    "zip": alt_groups_zip,
    "leetcode": Solution().numberOfAlternatingGroups,
    "leetcode_c2": SolutionExtended().numberOfAlternatingGroups,
    "leetcode_c3": SolutionEdges().numberOfAlternatingGroups,
    "leetcode_c4": SolutionUnroll().numberOfAlternatingGroups,
    "leetcode_c5": SolutionZip().numberOfAlternatingGroups,
}


def check_all(colors, expected, label):
    for key, fn in impls.items():
        actual = fn(list(colors))
        assert actual == expected, (label, key, colors, actual, expected)
    print("PASS", label)


for idx, (colors, expected) in enumerate(examples, 1):
    check_all(colors, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2


### 12.1. Các trường hợp biên

In [21]:
check_all([0, 1, 0], 1, "n = 3: chỉ nhóm có giữa 1")
check_all([0, 0, 0], 0, "toàn đỏ")
check_all([1, 0, 1], 1, "n = 3, giữa 0")
check_all([0, 0, 1], 1, "n = 3 không hằng: viên khác màu là giữa của một nhóm đi qua chỗ nối")
check_all([0, 1, 0, 1], 4, "n = 4 xen kẽ hoàn toàn: mọi vị trí đều là nhóm")
check_all([0, 1, 0, 1, 0], 3, "n = 5 lẻ: chỗ nối hai viên 0, 0 phá nhóm")
check_all([0, 1] * 50, 100, "n = 100 xen kẽ hoàn toàn")
check_all([1] * 100, 0, "n = 100 cùng màu")
check_all([0, 1, 1, 0, 1, 0], 2, "bài tính tay trong mục 17")
check_all([0, 1, 0, 0, 1, 0], 2, "hai viên 1 ở giữa hai viên 0: (0,1,0) và (0,1,0)")
check_all([1, 0, 0, 0, 1, 1], 0, "nhiều đoạn bằng nhau liên tiếp")

PASS n = 3: chỉ nhóm có giữa 1
PASS toàn đỏ
PASS n = 3, giữa 0
PASS n = 3 không hằng: viên khác màu là giữa của một nhóm đi qua chỗ nối
PASS n = 4 xen kẽ hoàn toàn: mọi vị trí đều là nhóm
PASS n = 5 lẻ: chỗ nối hai viên 0, 0 phá nhóm
PASS n = 100 xen kẽ hoàn toàn
PASS n = 100 cùng màu
PASS bài tính tay trong mục 17
PASS hai viên 1 ở giữa hai viên 0: (0,1,0) và (0,1,0)
PASS nhiều đoạn bằng nhau liên tiếp


### 12.2. Phản ví dụ cho các cách làm sai

Bốn lỗi phổ biến (chi tiết ở mục 13): (a) quên vòng (không xét nhóm đi qua điểm nối); (b) đọc `colors[i + 1]` ở cuối mảng gây `IndexError`; (c) chỉ kiểm tra hai đầu bằng nhau mà quên "khác giữa"; (d) đếm trùng khi trải vòng thành `2n`.

In [22]:
def wrong_no_wrap(colors):
    return sum(colors[i] != colors[i + 1] and colors[i + 1] != colors[i + 2] for i in range(len(colors) - 2))   # SAI: không vòng


def wrong_ends_equal_only(colors):
    n = len(colors)
    return sum(colors[i] == colors[(i + 2) % n] for i in range(n))                                           # SAI: thiếu "khác giữa"


def wrong_double_count(colors, k=3):
    n = len(colors)
    ans = cnt = 0
    for i in range(2 * n):
        if i > 0 and colors[i % n] == colors[(i - 1) % n]:
            cnt = 1
        else:
            cnt += 1
        if cnt >= k:                                                                                         # SAI: thiếu điều kiện i >= n
            ans += 1
    return ans


def wrong_index_error(colors):
    n = len(colors)
    return sum(colors[i] != colors[i + 1] and colors[i + 1] != colors[(i + 2) % n] for i in range(n))      # SAI: i + 1 vượt biên


print(wrong_no_wrap([0, 1, 0, 0, 1]), "!=", Solution().numberOfAlternatingGroups([0, 1, 0, 0, 1]))
print(wrong_ends_equal_only([1, 1, 1]), "!=", Solution().numberOfAlternatingGroups([1, 1, 1]))
print(wrong_double_count([0, 1, 0, 0, 1]), "!=", Solution().numberOfAlternatingGroups([0, 1, 0, 0, 1]))
assert wrong_no_wrap([0, 1, 0, 0, 1]) != 3
assert wrong_ends_equal_only([1, 1, 1]) != 0
assert wrong_double_count([0, 1, 0, 0, 1]) != 3
try:
    wrong_index_error([0, 1, 0, 0, 1])
    raise AssertionError("đáng lẽ phải lỗi")
except IndexError as e:
    print("IndexError:", e)

1 != 3
3 != 0
4 != 3
IndexError: list index out of range


### 12.3. Đối chiếu ngẫu nhiên (có seed) và vét cạn

Đầu vào hợp lệ: `n` từ `3` đến `100`, mỗi phần tử `0` hoặc `1`. Ta kiểm tra: (1) **ngẫu nhiên** `3000` mảng với seed cố định so `10` cài đặt; (2) **vét cạn** mọi mảng `0/1` độ dài `3..12` (`2^3 + ... + 2^12` mảng); (3) **tính chất**: đảo màu (`0 ↔ 1`) hoặc **xoay vòng** mảng không đổi đáp án.

In [23]:
rng = random.Random(3206)
for trial in range(3000):
    n_len = rng.choice([3, 4, 5, 6, 7, 10, 20, 50, 99, 100])
    colors = [rng.randint(0, 1) for _ in range(n_len)]
    ref = alt_groups_modulo(colors)
    for key, fn in impls.items():
        assert fn(list(colors)) == ref, (key, colors)
    shift = rng.randint(0, n_len - 1)
    assert alt_groups_modulo(colors[shift:] + colors[:shift]) == ref                # xoay vòng không đổi
    assert alt_groups_modulo([1 - x for x in colors]) == ref                        # đảo màu không đổi
print("PASS 3000 mảng ngẫu nhiên (kèm xoay vòng và đảo màu)")

from itertools import product

total = 0
for n_len in range(3, 13):
    for colors in product((0, 1), repeat=n_len):
        colors = list(colors)
        ref = alt_groups_modulo(colors)
        assert alt_groups_unroll(colors) == ref
        assert alt_groups_edges(colors) == ref
        assert Solution().numberOfAlternatingGroups(colors) == ref
        total += 1
print("PASS vét cạn", total, "mảng 0/1 độ dài 3..12")

PASS 3000 mảng ngẫu nhiên (kèm xoay vòng và đảo màu)


PASS vét cạn 8184 mảng 0/1 độ dài 3..12


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên "vòng tròn" (không xét nhóm qua điểm nối)

In [24]:
def sai_khong_vong(colors):
    count = 0
    for i in range(1, len(colors) - 1):                    # chỉ xét viên giữa ở vị trí 1..n-2
        if colors[i] != colors[i - 1] and colors[i] != colors[i + 1]:
            count += 1
    return count


print("sai:", sai_khong_vong([0, 1, 0, 0, 1]), "| đúng:", Solution().numberOfAlternatingGroups([0, 1, 0, 0, 1]))
print("sai:", sai_khong_vong([0, 1, 0, 1]), "| đúng:", Solution().numberOfAlternatingGroups([0, 1, 0, 1]))

sai: 1 | đúng: 3
sai: 2 | đúng: 4


**Vì sao sai:** vòng tròn có `n` viên giữa (mỗi viên đều có hai hàng xóm), nhưng vòng `1..n-2` bỏ mất viên `0` (hàng xóm trái là viên cuối) và viên `n-1` (hàng xóm phải là viên đầu). **Cách sửa:** dùng `% n`, chỉ số âm, hoặc kéo dài mảng (`colors + colors[:2]`).

### Lỗi 2: `IndexError` khi đọc `colors[i + 1]` ở cuối

In [25]:
def sai_index_error(colors):
    n = len(colors)
    count = 0
    for i in range(n):
        if colors[i] != colors[i + 1] and colors[i + 1] != colors[(i + 2) % n]:     # SAI: i + 1 = n khi i = n - 1
            count += 1
    return count


try:
    sai_index_error([0, 1, 0, 0, 1])
except IndexError as e:
    print("IndexError:", e)
print("đúng:", Solution().numberOfAlternatingGroups([0, 1, 0, 0, 1]))

IndexError: list index out of range
đúng: 3


**Vì sao sai:** khi `i = n - 1`, `i + 1 = n` vượt chỉ số hợp lệ lớn nhất `n - 1`. Lỗi này hay gặp ngay khi vừa nhớ ra phải xử lý vòng tròn nhưng chỉ sửa một trong hai chỉ số. **Cách sửa:** `colors[(i + 1) % n]` cho **mọi** chỉ số có thể vượt biên (hoặc kéo dài mảng).

### Lỗi 3: Chỉ kiểm tra hai đầu bằng nhau, quên "khác giữa"

In [26]:
def sai_thieu_dieu_kien(colors):
    n = len(colors)
    return sum(colors[i] == colors[(i + 2) % n] for i in range(n))            # SAI: [1,1,1] có hai đầu bằng nhau nhưng giữa cũng bằng


print("sai:", sai_thieu_dieu_kien([1, 1, 1]), "| đúng:", Solution().numberOfAlternatingGroups([1, 1, 1]))

sai: 3 | đúng: 0


**Vì sao sai:** nhóm xen kẽ cần **cả hai** điều: hai đầu bằng nhau **và** khác viên giữa. Chỉ kiểm tra hai đầu bằng nhau đếm cả nhóm `(1, 1, 1)`. **Cách sửa:** `b != a and b != c`, hoặc `a == c and a != b`.

### Lỗi 4: Đếm trùng khi trải vòng thành `2n`

In [27]:
def sai_dem_trung(colors, k=3):
    n = len(colors)
    ans = cnt = 0
    for i in range(2 * n):
        if i > 0 and colors[i % n] == colors[(i - 1) % n]:
            cnt = 1
        else:
            cnt += 1
        if cnt >= k:                       # SAI: đếm cả nửa đầu
            ans += 1
    return ans


print("sai:", sai_dem_trung([0, 1, 0, 0, 1]), "| đúng:", Solution().numberOfAlternatingGroups([0, 1, 0, 0, 1]))

sai: 4 | đúng: 3


**Vì sao sai:** khi trải thành `2n`, mỗi viên của vòng xuất hiện **hai lần** (vị trí `i` và `i + n`). Đếm ở cả hai nửa làm các nhóm không đi qua chỗ nối bị tính hai lần (và nhóm ở nửa đầu có thể chưa "đóng vòng" đúng). **Cách sửa:** chỉ đếm khi `i >= n`, mỗi viên đóng vai viên cuối của nhóm đúng một lần.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**Vòng tròn / mảng đầu nối cuối**" ⇒ dùng `(i + 1) % n`, chỉ số âm, nối đầu mảng vào cuối, hoặc **trải thành `2n`**.
- Chỉ có **hai giá trị** (0/1) ⇒ "khác" và "bằng" chuyển qua lại được: `b != a and b != c` ⇔ `a == c != b`.
- "Đoạn xen kẽ liên tiếp dài `>= k`" ⇒ giữ `cnt`: gặp hai phần tử **bằng nhau** thì `cnt = 1`, ngược lại `cnt += 1`; đếm khi `cnt >= k`.
- Với mảng vòng tròn, **xoay vòng không đổi đáp án**: dùng làm test kiểm tra nhanh.
- Luôn thử `n` nhỏ nhất (`n = 3`) và các mảng toàn một màu / xen kẽ hoàn toàn.
- Đếm theo **vị trí bắt đầu (hoặc kết thúc)** để mỗi nhóm đúng một lần.

## 15. Biến thể của bài

1. **Nhóm dài `k`** (Alternating Groups II, 3208): đếm các đoạn `k` viên liên tiếp xen kẽ trong vòng tròn; chính là Cách 4 với `cnt >= k`.
2. **Truy vấn thay đổi màu** (Alternating Groups III, 3245): cập nhật màu và đếm nhóm có độ dài cho trước; cần cấu trúc dữ liệu (Fenwick tree).
3. **Liệt kê các nhóm:** trả về danh sách chỉ số bắt đầu thay vì chỉ đếm.
4. **Mảng thẳng, không vòng:** bỏ điều kiện nối đầu cuối (đếm các bộ ba xen kẽ trong mảng thường).
5. Bài liên quan: **Count Alternating Subarrays (3101)**, **Count Hills and Valleys in an Array (2210)**, **Check if Array Is Sorted and Rotated (1752)**.

In [28]:
def alt_group_starts(colors):
    n = len(colors)
    return [i for i in range(n) if colors[(i + 1) % n] != colors[i] and colors[(i + 1) % n] != colors[(i + 2) % n]]


def alt_groups_linear(colors):
    """Biến thể: mảng thẳng, không nối đầu-cuối."""
    return sum(colors[i + 1] != colors[i] and colors[i + 1] != colors[i + 2] for i in range(len(colors) - 2))


print(alt_group_starts([0, 1, 0, 0, 1]))          # [0, 3, 4]
print(alt_groups_linear([0, 1, 0, 0, 1]))         # 1
print(alt_groups_unroll([0, 1, 0, 1, 1, 0], k=4))  # nhóm 4 viên liên tiếp xen kẽ trong vòng tròn

[0, 3, 4]
1
1


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Chỉ số `%` | ba viên `i, i+1, i+2` theo `% n` | `O(n)` | `O(1)` | bám sát đề |
| 2. Kéo dài mảng `+2` | dán hai viên đầu vào cuối, bỏ `%` | `O(n)` | `O(n)` | **nên nộp**, dễ đọc |
| 3. Mảng cạnh `d` | nhóm ⇔ hai cạnh liên tiếp cùng khác màu | `O(n)` | `O(n)` | tách bài thành hai bước |
| 4. Trải `2n` + `cnt` | đếm đoạn xen kẽ dài `>= k` | `O(n)` | `O(1)` | tổng quát cho `k` bất kỳ |
| 5. `zip` ba list dịch | ba lớp xếp chồng | `O(n)` | `O(n)` | code ngắn, kiểu Python |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `[0, 1, 0, 0, 1][-1]`, `(4 + 1) % 5`, `(4 + 2) % 5`, `[0, 1, 0] + [0, 1, 0][:2]`, `Solution().numberOfAlternatingGroups([1, 1, 1])`.

**Bài 2 (tính tay).** Với `colors = [0, 1, 1, 0, 1, 0]` (`n = 6`): liệt kê sáu nhóm bắt đầu ở `i = 0..5` (nhớ đi vòng ở cuối) và cho biết nhóm nào xen kẽ, tổng số nhóm.

**Bài 3 (điền TODO).** Hoàn thành `alt_groups_todo(colors)` theo Cách 1 (chỉ số `%`).

In [29]:
def alt_groups_todo(colors):
    # TODO 1: n = len(colors); count = 0
    # TODO 2: for i in range(n): a = colors[i]; b = colors[(i + 1) % n]; c = colors[(i + 2) % n]
    # TODO 3: nếu b != a và b != c thì count += 1
    # TODO 4: trả về count
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `alt_groups_doubling(colors)` **chỉ dùng** `colors + colors` (nối đôi cả mảng) và một vòng `for i in range(n)` đọc `d[i], d[i+1], d[i+2]`; không dùng `%`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `alt_group_starts(colors)` trả về **danh sách chỉ số bắt đầu** của các nhóm xen kẽ (tăng dần). (b) Viết `count_groups_k(colors, k)` đếm số nhóm `k` viên liên tiếp xen kẽ trong vòng tròn (`3 <= k <= n`), kiểm tra lại bằng cách `k = 3` phải khớp `Solution`.

In [30]:
def alt_groups_doubling(colors):
    # TODO: d = colors + colors; đếm i trong range(n) thỏa d[i+1] != d[i] và d[i+1] != d[i+2]
    raise NotImplementedError


def alt_group_starts_todo(colors):
    # TODO: trả về list các chỉ số i bắt đầu nhóm xen kẽ
    raise NotImplementedError


def count_groups_k(colors, k):
    # TODO: tổng quát hóa Cách 4 (cnt >= k) hoặc kiểm tra từng cửa sổ k viên
    raise NotImplementedError

# KHÔNG gọi các hàm TODO này khi Run All.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `1`, `0`, `1`, `[0, 1, 0, 0, 1]`, `0`. Bài 3: `b != a and b != c`.
- **Mức 2 (vừa):** Bài 2: `i=0`: (0,1,1) ✗; `i=1`: (1,1,0) ✗; `i=2`: (1,0,1) ✓; `i=3`: (0,1,0) ✓; `i=4`: (1,0,0) ✗ (viên 4, 5, 0); `i=5`: (0,0,1) ✗ (viên 5, 0, 1) ⇒ `2`. Bài 4: `d = colors + colors` dài `2n`, nên `d[i + 2]` với `i <= n - 1` hợp lệ.
- **Mức 3 (sâu):** Bài 5(a): trả `i` thay vì `count += 1`. Bài 5(b): với mỗi vị trí bắt đầu `i`, kiểm tra `colors[(i + t) % n] != colors[(i + t + 1) % n]` cho `t = 0..k-2`; hoặc dùng Cách 4 với `cnt >= k` (cần `k <= n`).

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [31]:
# Bài 1
assert [0, 1, 0, 0, 1][-1] == 1
assert (4 + 1) % 5 == 0
assert (4 + 2) % 5 == 1
assert [0, 1, 0] + [0, 1, 0][:2] == [0, 1, 0, 0, 1]
assert Solution().numberOfAlternatingGroups([1, 1, 1]) == 0

# Bài 2
c2 = [0, 1, 1, 0, 1, 0]
groups = [(c2[i], c2[(i + 1) % 6], c2[(i + 2) % 6]) for i in range(6)]
assert groups == [(0, 1, 1), (1, 1, 0), (1, 0, 1), (0, 1, 0), (1, 0, 0), (0, 0, 1)]
assert Solution().numberOfAlternatingGroups(c2) == 2


# Bài 3
def alt_groups_todo_ans(colors):
    n = len(colors)
    count = 0
    for i in range(n):
        a = colors[i]
        b = colors[(i + 1) % n]
        c = colors[(i + 2) % n]
        if b != a and b != c:
            count += 1
    return count


for colors, expected in examples:
    assert alt_groups_todo_ans(colors) == expected


# Bài 4
def alt_groups_doubling_ans(colors):
    n = len(colors)
    d = colors + colors
    return sum(1 for i in range(n) if d[i + 1] != d[i] and d[i + 1] != d[i + 2])


rng = random.Random(7)
for _ in range(500):
    arr = [rng.randint(0, 1) for _ in range(rng.randint(3, 30))]
    assert alt_groups_doubling_ans(arr) == Solution().numberOfAlternatingGroups(arr)


# Bài 5
def alt_group_starts_ans(colors):
    n = len(colors)
    return [i for i in range(n)
            if colors[(i + 1) % n] != colors[i] and colors[(i + 1) % n] != colors[(i + 2) % n]]


def count_groups_k_ans(colors, k):
    n = len(colors)
    total = 0
    for i in range(n):
        if all(colors[(i + t) % n] != colors[(i + t + 1) % n] for t in range(k - 1)):
            total += 1
    return total


assert alt_group_starts_ans([0, 1, 0, 0, 1]) == [0, 3, 4]
assert len(alt_group_starts_ans(c2)) == 2
for _ in range(500):
    n_len = rng.randint(3, 30)
    arr = [rng.randint(0, 1) for _ in range(n_len)]
    assert count_groups_k_ans(arr, 3) == Solution().numberOfAlternatingGroups(arr)
    assert len(alt_group_starts_ans(arr)) == Solution().numberOfAlternatingGroups(arr)
    k = rng.randint(3, n_len)
    assert count_groups_k_ans(arr, k) == alt_groups_unroll(arr, k)
print("Tất cả đáp án bài tập đều PASS")

Tất cả đáp án bài tập đều PASS


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (mỗi vị trí bắt đầu là một nhóm; dán hai viên đầu vào cuối để đóng vòng)

```python
class Solution:
    def numberOfAlternatingGroups(self, colors: List[int]) -> int:
        n = len(colors)
        ext = colors + colors[:2]
        return sum(ext[i + 1] != ext[i] and ext[i + 1] != ext[i + 2] for i in range(n))
```

**Checklist**

- [ ] Đã xét **đủ `n` nhóm** (kể cả hai nhóm đi qua chỗ nối đầu-cuối)?
- [ ] Mọi chỉ số vượt biên đã qua `% n` hoặc nằm trong mảng kéo dài?
- [ ] Điều kiện là "giữa khác **cả hai** bên" (`b != a and b != c`)?
- [ ] Nếu trải `2n`: chỉ đếm khi `i >= n` (không đếm trùng)?
- [ ] Đã thử `[1,1,1]` (`0`), `[0,1,0,0,1]` (`3`), `[0,1,0]`, và mảng xen kẽ hoàn toàn?